In [ ]:
import os
import io
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import httpx
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.2f}".format)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
(DATA_DIR / "ct_boundaries").mkdir(exist_ok=True)
(DATA_DIR / "census_profile").mkdir(exist_ok=True)

(DATA_DIR / "cimd").mkdir(exist_ok=True)

ARCGIS_TOKEN = os.getenv("ARCGIS_TOKEN", "")
if not ARCGIS_TOKEN:
    print("⚠️  ARCGIS_TOKEN not set — B1/B2 Esri sources will be skipped gracefully")
else:
    print("✅ ARCGIS_TOKEN loaded")

print("Setup complete. DATA_DIR:", DATA_DIR.resolve())

: 

In [ ]:
# A1 — StatsCan Census Tract boundaries 2021
# Cartographic boundary file (simplified), national coverage, ~15 MB zip
CT_URL = (
    "https://www12.statcan.gc.ca/census-recensement/2021/geo/sip-pis/"
    "boundary-limites/files-fichiers/lct_000b21a_e.zip"
)
CT_ZIP = DATA_DIR / "ct_boundaries" / "lct_000b21a_e.zip"

if not CT_ZIP.exists():
    print("Downloading CT boundaries (~15 MB)...")
    with httpx.Client(follow_redirects=True, timeout=120) as client:
        r = client.get(CT_URL)
        r.raise_for_status()
    CT_ZIP.write_bytes(r.content)
    print(f"Saved to {CT_ZIP}")
else:
    print(f"Using cached {CT_ZIP}")

with zipfile.ZipFile(CT_ZIP) as z:
    z.extractall(DATA_DIR / "ct_boundaries")

# Read — geopandas can read the shapefile directly from the extracted dir
shp_files = list((DATA_DIR / "ct_boundaries").glob("*.shp"))
assert shp_files, "No .shp file found after extraction"
gdf_ct = gpd.read_file(shp_files[0])

print("Raw columns:", gdf_ct.columns.tolist())
print("Raw CRS:", gdf_ct.crs)
print("Raw shape:", gdf_ct.shape)

# Filter to CMAs 35535 (Toronto CMA — covers Mississauga + Brampton) and 35537 (Hamilton)
# CMAUID column name may be 'CMAUID' or 'CMAPUID' — check print above and adjust
# Detect the CMA UID column name (varies between StatsCan releases)
cma_col = "CMAUID" if "CMAUID" in gdf_ct.columns else "CMAPUID"
if cma_col not in gdf_ct.columns:
    raise KeyError(f"Neither CMAUID nor CMAPUID found. Columns: {gdf_ct.columns.tolist()}")
gdf_ct = gdf_ct[gdf_ct[cma_col].astype(str).isin(["35535", "35537"])].copy()
gdf_ct = gdf_ct.to_crs("EPSG:4326").reset_index(drop=True)

print(f"\nFiltered to {len(gdf_ct)} CTs in CMAs 35535 + 35537")
gdf_ct[["CTUID", "CTNAME", cma_col, "CMANAME"]].head(3)


In [ ]:
# ASSERTION — run after the fetch cell to verify
assert "gdf_ct" in dir(), "gdf_ct not defined — run the fetch cell"
assert len(gdf_ct) >= 350, f"Expected ≥350 CTs in scope, got {len(gdf_ct)}"
assert gdf_ct.crs.to_epsg() == 4326, f"Expected EPSG:4326, got {gdf_ct.crs}"
assert "CTUID" in gdf_ct.columns, "CTUID column missing"
assert gdf_ct.geometry.notnull().all(), "Null geometries found"
print(f"✅ A1 assertions pass — {len(gdf_ct)} CTs, CRS: {gdf_ct.crs}")


In [ ]:
# A2 — Census Profile 2021, CT level (English, national)
# Large file (~1.5 GB unzipped). Downloads the zip (~200 MB), extracts, reshapes.
CENSUS_URL = (
    "https://www12.statcan.gc.ca/census-recensement/2021/dp-pd/prof/details/"
    "download-telecharger/comp/GetFile.cfm?Lang=E&FILETYPE=CSV&GEONO=044"
)
CENSUS_ZIP = DATA_DIR / "census_profile" / "98-401-X2021044_English_CSV.zip"
CENSUS_CSV = DATA_DIR / "census_profile" / "98-401-X2021044_English_CSV_data.csv"

if not CENSUS_CSV.exists():
    if not CENSUS_ZIP.exists():
        print("Downloading Census Profile CSV (~200 MB)...")
        with httpx.Client(follow_redirects=True, timeout=300) as client:
            r = client.get(CENSUS_URL)
            r.raise_for_status()
        CENSUS_ZIP.write_bytes(r.content)
        print("Download complete")
    print("Extracting...")
    with zipfile.ZipFile(CENSUS_ZIP) as z:
        z.extractall(DATA_DIR / "census_profile")
    print("Extracted")
else:
    print(f"Using cached {CENSUS_CSV}")

# Read only the rows we need to avoid loading 1.5 GB into memory
# The CSV is long-format: one row per (GEO_CODE, CHARACTERISTIC_ID)
# We filter to the 8 characteristic IDs we need
TARGET_CHARS = {1, 133, 1680, 1681, 1682, 1820, 1838, 1839}
CHAR_LABELS  = {
    1:    "population",
    133:  "median_income",
    1682: "renter_count",
    1681: "owner_count",
    1820: "total_period_construct",
    1838: "dwell_pre1960",
    1839: "dwell_1961_1980",
}

chunks = []
for chunk in pd.read_csv(CENSUS_CSV, chunksize=50_000, low_memory=False, encoding="latin-1"):
    chunk.columns = chunk.columns.str.strip()
    mask = chunk["CHARACTERISTIC_ID"].isin(TARGET_CHARS)
    chunks.append(chunk[mask])

df_long = pd.concat(chunks, ignore_index=True)
print(f"Filtered rows: {len(df_long)}")
print("Columns:", df_long.columns.tolist()[:10])

# Validate expected column names before pivoting (FIX 1)
geo_col = "ALT_GEO_CODE"
val_col = "C1_COUNT_TOTAL"
if geo_col not in df_long.columns:
    # Try alternate geo column names
    for candidate in ["GEO_CODE (POR)", "GEO_CODE", "DGUID"]:
        if candidate in df_long.columns:
            geo_col = candidate
            break
    else:
        raise KeyError(f"No geo column found. Columns: {df_long.columns.tolist()}")
if val_col not in df_long.columns:
    for candidate in ["C1_COUNT_TOTAL", "C1_COUNT_TOTAL:", "DATA_QUALITY_FLAG"]:
        if candidate in df_long.columns:
            val_col = candidate
            break
    else:
        raise KeyError(f"No value column found. Columns: {df_long.columns.tolist()}")
print(f"Using geo_col='{geo_col}', val_col='{val_col}'")

df_wide = (
    df_long[df_long["CHARACTERISTIC_ID"].isin(CHAR_LABELS)]
    .pivot_table(index=geo_col, columns="CHARACTERISTIC_ID", values=val_col, aggfunc="first")
    .reset_index()
    .rename(columns={**{k: v for k, v in CHAR_LABELS.items()}, geo_col: "CTUID"})
)

# Post-pivot source column validation (FIX 2)
expected_wide_cols = ["owner_count", "renter_count", "dwell_pre1960", "dwell_1961_1980", "total_period_construct"]
missing_wide = [c for c in expected_wide_cols if c not in df_wide.columns]
if missing_wide:
    print(f"⚠️  Missing derived columns (characteristic IDs may differ): {missing_wide}")
    print(f"   Available columns: {df_wide.columns.tolist()}")

df_wide["CTUID"] = df_wide["CTUID"].astype(str).str.zfill(7)

# Derived columns
df_wide["total_dwellings"] = df_wide.get("owner_count", pd.Series(0)) + df_wide.get("renter_count", pd.Series(0))
df_wide["pct_renters"]  = df_wide["renter_count"]  / df_wide["total_dwellings"].replace(0, np.nan)
df_wide["pct_pre1980"]  = (
    (df_wide.get("dwell_pre1960", pd.Series(0)).fillna(0) + df_wide.get("dwell_1961_1980", pd.Series(0)).fillna(0))
    / df_wide["total_period_construct"].replace(0, np.nan)
)
df_wide["median_income"] = pd.to_numeric(df_wide.get("median_income", pd.Series(dtype=float)), errors="coerce")

df_census = df_wide[["CTUID","population","median_income","pct_renters","pct_pre1980"]].copy()
print(f"\ndf_census shape: {df_census.shape}")
df_census.head(3)

In [ ]:
# ASSERTION
assert "df_census" in dir(), "df_census not defined"
assert {"CTUID", "population", "median_income", "pct_renters", "pct_pre1980"}.issubset(df_census.columns),     f"Missing columns. Got: {df_census.columns.tolist()}"
assert len(df_census) >= 350, f"Expected ≥350 rows, got {len(df_census)}"
null_pct = df_census[["median_income", "pct_renters", "pct_pre1980"]].isnull().mean()
assert (null_pct < 0.10).all(), f"High null rate in core columns:\n{null_pct}"
print(f"✅ A2 assertions pass — {len(df_census)} CTs")
print(df_census[["CTUID","median_income","pct_renters","pct_pre1980"]].describe())

In [ ]:
# A3 — Canadian Index of Multiple Deprivation (CIMD) 2021
# StatsCan catalogue 45-20-0001-01
CIMD_URL = "https://www150.statcan.gc.ca/n1/pub/45-20-0001/2021001/CIMD-ICMD.zip"
CIMD_ZIP = DATA_DIR / "cimd" / "CIMD-ICMD.zip"

if not CIMD_ZIP.exists():
    print("Downloading CIMD (~5 MB)...")
    with httpx.Client(follow_redirects=True, timeout=60) as client:
        r = client.get(CIMD_URL)
        r.raise_for_status()
    CIMD_ZIP.write_bytes(r.content)
    print("Downloaded")
else:
    print(f"Using cached {CIMD_ZIP}")

with zipfile.ZipFile(CIMD_ZIP) as z:
    z.extractall(DATA_DIR / "cimd")

csv_files = list((DATA_DIR / "cimd").glob("*.csv"))
assert csv_files, f"No CSV found after extraction. Files: {list((DATA_DIR / 'cimd').iterdir())}"
df_cimd_raw = pd.read_csv(csv_files[0], encoding="latin-1")
print("Columns:", df_cimd_raw.columns.tolist())
print(df_cimd_raw.head(2))

# Detect CTUID column — may be CT_UID, CTUID, or CTNAM
ct_col = next((c for c in df_cimd_raw.columns if "CT" in c.upper() and "UID" in c.upper()), None)
if ct_col is None:
    raise KeyError(f"No CT UID column found. Columns: {df_cimd_raw.columns.tolist()}")
print(f"Using CT column: '{ct_col}'")

# Detect CIMD sub-index columns — percentile columns typically end in _PCTL or _PCT
CIMD_COL_MAP = {}
for target, candidates in [
    ("cimd_residential_instability", ["RINS_PCTL", "RINS_PCT", "RESIDENTIAL_INSTABILITY_PCTL"]),
    ("cimd_economic_dependency",     ["ECON_PCTL", "ECON_PCT", "ECONOMIC_DEPENDENCY_PCTL"]),
    ("cimd_ethnocultural_composition",["ETHN_PCTL", "ETHN_PCT", "ETHNOCULTURAL_COMPOSITION_PCTL"]),
    ("cimd_situational_vulnerability",["SITU_PCTL", "SITU_PCT", "SITUATIONAL_VULNERABILITY_PCTL"]),
]:
    found = next((c for c in candidates if c in df_cimd_raw.columns), None)
    if found:
        CIMD_COL_MAP[found] = target
    else:
        print(f"⚠️  Could not find column for {target}. Available: {df_cimd_raw.columns.tolist()}")

if not CIMD_COL_MAP:
    raise KeyError(
        f"No CIMD sub-index columns found in {csv_files[0].name}. "
        f"Expected one of: RINS_PCTL, ECON_PCTL, ETHN_PCTL, SITU_PCTL. "
        f"Available columns: {df_cimd_raw.columns.tolist()}"
    )

df_cimd = df_cimd_raw.rename(columns={ct_col: "CTUID", **CIMD_COL_MAP})
keep = ["CTUID"] + list(CIMD_COL_MAP.values())
df_cimd = df_cimd[[c for c in keep if c in df_cimd.columns]].copy()
df_cimd["CTUID"] = df_cimd["CTUID"].astype(str).str.zfill(7)

for col in [c for c in df_cimd.columns if c != "CTUID"]:
    df_cimd[col] = pd.to_numeric(df_cimd[col], errors="coerce") / 100.0

print(f"\ndf_cimd shape: {df_cimd.shape}")
df_cimd.head(3)


In [ ]:
# ASSERTION
assert "df_cimd" in dir(), "df_cimd not defined"
required = {"CTUID","cimd_residential_instability","cimd_economic_dependency",
            "cimd_ethnocultural_composition","cimd_situational_vulnerability"}
assert required.issubset(df_cimd.columns), f"Missing cols. Got: {df_cimd.columns.tolist()}"
assert len(df_cimd) >= 350
print(f"✅ A3 assertions pass — {len(df_cimd)} rows")
print(df_cimd[list(required - {'CTUID'})].describe())


In [ ]:
# A8 — Alectra service area
# ArcGIS Online item ID: 8eba357e1b124587884bccb724743c4c
ITEM_URL = "https://www.arcgis.com/sharing/rest/content/items/8eba357e1b124587884bccb724743c4c?f=json"
with httpx.Client(follow_redirects=True, timeout=30) as client:
    meta = client.get(ITEM_URL).json()

print("Item type:", meta.get("type"))
print("URL:", meta.get("url"))
service_url = meta.get("url", "").rstrip("/")

if not service_url:
    raise ValueError("Could not resolve service URL from item metadata. Check item ID.")

# Query layer 0 for all features as GeoJSON
query_url = f"{service_url}/0/query"
params = {
    "f": "geojson",
    "where": "1=1",
    "outFields": "*",
    "returnGeometry": "true",
}
with httpx.Client(follow_redirects=True, timeout=60) as client:
    r = client.get(query_url, params=params)
    r.raise_for_status()

gdf_alectra = gpd.read_file(io.StringIO(r.text))
gdf_alectra = gdf_alectra.to_crs("EPSG:4326")
print(f"A8 fetched: {len(gdf_alectra)} features")
print("Columns:", gdf_alectra.columns.tolist())


In [ ]:
# ASSERTION
assert "gdf_alectra" in dir(), "gdf_alectra not defined"
assert gdf_alectra.crs.to_epsg() == 4326
assert len(gdf_alectra) >= 1, "No features returned"
assert gdf_alectra.geometry.notnull().all()
print(f"✅ A8 assertions pass — {len(gdf_alectra)} service area polygon(s)")
gdf_alectra.plot(figsize=(6, 6), edgecolor="black", facecolor="lightblue")
plt.title("Alectra Service Area"); plt.show()


In [ ]:
# C1 — Alectra live outage polygons
# Step 1: enumerate all layers to find the customer outage polygon layer
FEAT_SERVER = "https://services8.arcgis.com/wNDmObY7QplwZD9m/ArcGIS/rest/services/Outage_Details/FeatureServer"
with httpx.Client(follow_redirects=True, timeout=30) as client:
    layers_resp = client.get(f"{FEAT_SERVER}/layers?f=json")
    layers_data = layers_resp.json()

print("Available layers:")
for lyr in layers_data.get("layers", []):
    print(f"  id={lyr['id']}  name={lyr['name']}  geomType={lyr.get('geometryType','?')}")

# Auto-detect polygon layer most likely to be customer outage areas
# Prefer polygon geometry, name containing 'outage' or 'customer' (case-insensitive)
outage_layer_id = None
for lyr in layers_data.get("layers", []):
    name_lower = lyr["name"].lower()
    geom = lyr.get("geometryType", "")
    if "polygon" in geom.lower() and any(kw in name_lower for kw in ["outage","customer","area"]):
        outage_layer_id = lyr["id"]
        print(f"Auto-selected layer {outage_layer_id}: {lyr['name']}")
        break

if outage_layer_id is None:
    # Fall back to first polygon layer
    for lyr in layers_data.get("layers", []):
        if "polygon" in lyr.get("geometryType", "").lower():
            outage_layer_id = lyr["id"]
            print(f"Fallback to first polygon layer {outage_layer_id}: {lyr['name']}")
            break

if outage_layer_id is None:
    print("⚠️  No polygon layer found — using layer 0")
    outage_layer_id = 0

# Step 2: query for active outages
query_url = f"{FEAT_SERVER}/{outage_layer_id}/query"
params = {
    "f": "geojson",
    "where": "1=1",
    "outFields": "*",
    "returnGeometry": "true",
}
with httpx.Client(follow_redirects=True, timeout=30) as client:
    r = client.get(query_url, params=params)
    r.raise_for_status()

try:
    gdf_outages = gpd.read_file(io.StringIO(r.text))
    if len(gdf_outages) > 0:
        gdf_outages = gdf_outages.to_crs("EPSG:4326")
    print(f"C1 fetched: {len(gdf_outages)} active outages")
    if len(gdf_outages) > 0:
        print("Columns:", gdf_outages.columns.tolist())
except Exception as e:
    print(f"No active outages or parse error: {e}")
    gdf_outages = gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs="EPSG:4326")


In [ ]:
# ASSERTION
assert "gdf_outages" in dir(), "gdf_outages not defined"
assert isinstance(gdf_outages, gpd.GeoDataFrame)
assert len(gdf_outages) == 0 or gdf_outages.crs.to_epsg() == 4326
print(f"✅ C1 assertions pass — {len(gdf_outages)} active outage polygon(s) right now")
if len(gdf_outages) > 0:
    print(gdf_outages[["geometry"]].head(3))


In [ ]:
# C2 — Environment Canada GeoMet OGC API — hourly observations near study area
# Bounding box: Hamilton + Mississauga + Brampton approximately -80.5, 43.1, -79.2, 43.9
BBOX = "-80.5,43.1,-79.2,43.9"
OBS_URL = "https://api.weather.gc.ca/collections/climate-hourly/items"
params = {
    "f": "json",
    "bbox": BBOX,
    "limit": 100,
    "sortby": "-LOCAL_DATE",
}
gdf_weather = None
with httpx.Client(follow_redirects=True, timeout=30) as client:
    r = client.get(OBS_URL, params=params)
    if r.status_code == 200:
        try:
                    data = r.json()
                except ValueError as e:
                    print(f"Failed to parse JSON response: {e}")
                    data = {"features": []}
                features = data.get("features", [])
        print(f"Got {len(features)} observations from climate-hourly")
        if features:
            gdf_weather = gpd.GeoDataFrame.from_features(features, crs="EPSG:4326")
            temp_candidates = [c for c in gdf_weather.columns if "TEMP" in c.upper()]
            print("Temperature candidates:", temp_candidates)
            if temp_candidates:
                gdf_weather["temperature_c"] = pd.to_numeric(gdf_weather[temp_candidates[0]], errors="coerce")
            else:
                gdf_weather["temperature_c"] = np.nan
            gdf_weather["humidex"] = pd.to_numeric(
                gdf_weather.get("HUMIDEX", pd.Series(dtype=float)), errors="coerce"
            ) if "HUMIDEX" in gdf_weather.columns else np.nan
    else:
        print(f"climate-hourly returned {r.status_code}. Trying alternate collection...")
        # List available collections for diagnosis
        coll_r = client.get("https://api.weather.gc.ca/collections?f=json", timeout=30)
        if coll_r.status_code == 200:
            colls = [c["id"] for c in coll_r.json().get("collections", [])]
            print("Available collections (first 20):", colls[:20])

if gdf_weather is None or len(gdf_weather) == 0:
    print("No weather data — using empty GeoDataFrame")
    gdf_weather = gpd.GeoDataFrame(
        columns=["geometry","temperature_c","humidex"],
        geometry="geometry", crs="EPSG:4326"
    )
print(f"gdf_weather: {len(gdf_weather)} observations")


In [ ]:
# ASSERTION
assert "gdf_weather" in dir(), "gdf_weather not defined"
assert isinstance(gdf_weather, gpd.GeoDataFrame)
assert len(gdf_weather) == 0 or "temperature_c" in gdf_weather.columns, \
    f"Expected temperature_c. Got columns: {gdf_weather.columns.tolist()}"
print(f"✅ C2 assertions pass — {len(gdf_weather)} weather observation(s)")


In [ ]:
# B1/B2 — Esri Living Atlas EJ + Climate Hub heat vulnerability
# These require an ArcGIS Online token. Cells skip gracefully if token is absent.

gdf_ej = None
gdf_heat_vuln = None

if not ARCGIS_TOKEN:
    print("⚠️  Skipping B1/B2 — set ARCGIS_TOKEN env var and rerun to include Esri sources")
else:
    # B1: Living Atlas Environmental Justice — Canada
    B1_URL = os.getenv("B1_EJ_LAYER_URL", "")
    if B1_URL:
        params = {"f":"geojson","where":"1=1","outFields":"*","token":ARCGIS_TOKEN}
        with httpx.Client(follow_redirects=True, timeout=60) as client:
            r = client.get(f"{B1_URL}/query", params=params)
        gdf_ej = gpd.read_file(io.StringIO(r.text)).to_crs("EPSG:4326")
        print(f"B1 EJ: {len(gdf_ej)} features, columns: {gdf_ej.columns.tolist()[:8]}")
    else:
        print("⚠️  B1_EJ_LAYER_URL not set — skipping B1")

    # B2: Esri Canada Climate Hub — heat vulnerability
    B2_URL = os.getenv("B2_HEAT_LAYER_URL", "")
    if B2_URL:
        params = {"f":"geojson","where":"1=1","outFields":"*","token":ARCGIS_TOKEN}
        with httpx.Client(follow_redirects=True, timeout=60) as client:
            r = client.get(f"{B2_URL}/query", params=params)
        gdf_heat_vuln = gpd.read_file(io.StringIO(r.text)).to_crs("EPSG:4326")
        print(f"B2 heat: {len(gdf_heat_vuln)} features")
    else:
        print("⚠️  B2_HEAT_LAYER_URL not set — skipping B2")

print("B1/B2 fetch complete. gdf_ej:", "loaded" if gdf_ej is not None else "skipped",
      "| gdf_heat_vuln:", "loaded" if gdf_heat_vuln is not None else "skipped")
